# Rating Attribution Notebook

Traces a single player performance through every step of the rating pipeline,
showing exactly where the final rating comes from.

**Usage:** paste `MATCH_DATA` and `PERFORMANCE` at the top of cell 1, then run all cells.

In [31]:
# ── INPUTS — paste your dicts here ───────────────────────────────────────────

TEAM_NAME = "Valencia CF"

MATCH_DATA = {
            "in_game_date": "2031-04-19T00:00:00",
            "half_length": 10,
            "competition": "La Liga",
            "home_team_name": "Rayo Vallecano",
            "away_team_name": "Valencia CF",
            "home_score": 1,
            "away_score": 1,
            "home_stats": {
                "possession": 8,
                "ball_recovery": 8,
                "shots": 7,
                "xg": 0.1,
                "passes": 174,
                "tackles": 32,
                "tackles_won": 25,
                "interceptions": 17,
                "saves": 5,
                "fouls_committed": 1,
                "offsides": 2,
                "corners": 2,
                "free_kicks": 7,
                "penalty_kicks": 0,
                "yellow_cards": 1
            },
            "away_stats": {
                "possession": 60,
                "ball_recovery": 7,
                "shots": 8,
                "xg": 2.7,
                "passes": 254,
                "tackles": 67,
                "tackles_won": 24,
                "interceptions": 19,
                "saves": 2,
                "fouls_committed": 7,
                "offsides": 0,
                "corners": 1,
                "free_kicks": 3,
                "penalty_kicks": 0,
                "yellow_cards": 1
            }
        }

PERFORMANCE = {
                "performance_type": "Outfield",
                "positions_played": [
                    "CB"
                ],
                "goals": 0,
                "assists": 0,
                "shots": 0,
                "shot_accuracy": 0,
                "passes": 14,
                "pass_accuracy": 100,
                "dribbles": 12,
                "dribble_success_rate": 100,
                "tackles": 7,
                "tackle_success_rate": 14,
                "offsides": 0,
                "fouls_committed": 2,
                "possession_won": 5,
                "possession_lost": 0,
                "minutes_played": 58,
                "distance_covered": 6.8,
                "distance_sprinted": 2.2,
                "player_id": 62
            }

In [32]:
from pathlib import Path
import sys, json, math
import numpy as np

project_root = Path("..").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.services.analytics.match_ratings_service import MatchRatingsService

with open(project_root / "config" / "performance_weights.json")    as f: weights    = json.load(f)
with open(project_root / "config" / "performance_means_stds.json") as f: means_stds = json.load(f)


class AttributionService(MatchRatingsService):
    """Subclass that captures every intermediate value during rating calculation."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attr: dict = {}

    def _apply_bayesian_smoothing(self, normalized_metrics, pos, minutes_played):
        self.attr["normalized_metrics"] = dict(normalized_metrics)
        result = super()._apply_bayesian_smoothing(normalized_metrics, pos, minutes_played)
        self.attr["p90_metrics"] = dict(result)
        return result

    def _calculate_z_scores(self, p90_metrics, pos_means_stds, normalized_metrics):
        result = super()._calculate_z_scores(p90_metrics, pos_means_stds, normalized_metrics)
        self.attr["z_scores"] = dict(result)
        self.attr["pos_means_stds"] = dict(pos_means_stds)
        return result

    def _calculate_dot_product(self, z_scores, weights):
        result = super()._calculate_dot_product(z_scores, weights)
        self.attr["dot_product"] = result
        self.attr["z_scores_at_dot"] = dict(z_scores)
        self.attr["weights_at_dot"] = list(weights)
        return result

    def _apply_mastery_bonus(self, z_scores, key_a, key_b, threshold, impact_scalar):
        za = z_scores.get(key_a, 0.0)
        zb = z_scores.get(key_b, 0.0)
        mn = min(za, zb)
        excess = mn - threshold
        fired = excess > 0
        bonus = 0.0
        if fired:
            bonus = min(excess, self.MASTERY_EXCESS_CAP) * self.MASTERY_WEIGHT * impact_scalar
        self.attr.setdefault("mastery_log", []).append({
            "key_a": key_a, "val_a": round(za, 4),
            "key_b": key_b, "val_b": round(zb, 4),
            "threshold": threshold, "min_z": round(mn, 4),
            "excess": round(max(0.0, excess), 4),
            "fired": fired, "bonus": round(bonus, 4),
        })
        return bonus

    def _apply_pos_modifiers(self, base_rating, z_scores, pos, opponent_goals, opponent_xg,
                             performance_metrics, minutes_played, impact_scalar,
                             isolation_multiplier=1.0):
        self.attr["pre_modifier"] = {
            "base_rating": base_rating, "pos": pos,
            "opponent_goals": opponent_goals, "opponent_xg": round(opponent_xg, 3),
            "minutes_played": minutes_played, "impact_scalar": round(impact_scalar, 4),
            "isolation_multiplier": round(isolation_multiplier, 4),
            "goals": performance_metrics.get("goals", 0),
            "assists": performance_metrics.get("assists", 0),
            "shots": performance_metrics.get("shots", 0),
            "pass_accuracy": performance_metrics.get("pass_accuracy", 0),
            "possession_lost": performance_metrics.get("possession_lost", 0),
        }
        self.attr["mastery_log"] = []
        bonus = super()._apply_pos_modifiers(
            base_rating=base_rating, z_scores=z_scores, pos=pos,
            opponent_goals=opponent_goals, opponent_xg=opponent_xg,
            performance_metrics=performance_metrics, minutes_played=minutes_played,
            impact_scalar=impact_scalar, isolation_multiplier=isolation_multiplier
        )
        self.attr["total_bonus"] = bonus
        return bonus

    def _calculate_match_supremacy_scalar(self, team_xg, xg_against):
        result = super()._calculate_match_supremacy_scalar(team_xg, xg_against)
        self.attr["supremacy_scalar"] = result
        self.attr["team_xg"] = team_xg
        self.attr["xg_against"] = xg_against
        return result

    def _calculate_tactical_isolation_multiplier(self, z_scores):
        result = super()._calculate_tactical_isolation_multiplier(z_scores)
        self.attr["isolation_multiplier"] = round(result, 4)
        return result


svc = AttributionService(weights, means_stds)
print("Service ready.")

Service ready.


In [33]:
# Run the service — this populates svc.attr with all intermediate values
final_rating = svc.calculate_outfield_rating(
    performance=PERFORMANCE,
    match_overview=MATCH_DATA,
    half_length=MATCH_DATA["half_length"],
    team_name=TEAM_NAME,
)

# Extract match context for display
is_home   = MATCH_DATA["home_team_name"] == TEAM_NAME
team_stats = MATCH_DATA["home_stats"] if is_home else MATCH_DATA["away_stats"]
opp_stats  = MATCH_DATA["away_stats"] if is_home else MATCH_DATA["home_stats"]
pos        = PERFORMANCE["positions_played"][0]
minutes    = PERFORMANCE["minutes_played"]
half_len   = MATCH_DATA["half_length"]
team_xg    = team_stats["xg"]
opp_xg     = opp_stats["xg"]
opp_goals  = (MATCH_DATA["away_score"] if is_home else MATCH_DATA["home_score"])

print(f"Match:    {MATCH_DATA['home_team_name']} vs {MATCH_DATA['away_team_name']} "
      f"({MATCH_DATA['home_score']}-{MATCH_DATA['away_score']})")
print(f"Player:   {PERFORMANCE['player_id']}  pos={pos}  minutes={minutes}")
print(f"Team xG:  {team_xg}   Opponent xG: {opp_xg}   Opponent goals: {opp_goals}")
print(f"Half length: {half_len} min  |  H_BASE: {svc.H_BASE}  |  time scalar: {svc.H_BASE/half_len}")
print(f"Possession: {team_stats['possession']}%")
print()
print(f"FINAL RATING: {final_rating}")

Match:    Rayo Vallecano vs Valencia CF (1-1)
Player:   62  pos=CB  minutes=58
Team xG:  2.7   Opponent xG: 0.1   Opponent goals: 1
Half length: 10 min  |  H_BASE: 10.0  |  time scalar: 1.0
Possession: 60%

FINAL RATING: 6.8


In [34]:
# ── Step 1: Half-length normalisation ────────────────────────────────────────
# The service applies ONE transformation at this stage:
#   volume stats × (H_BASE / half_length)
# Goals, assists, shots are NOT in vol_columns and are passed through raw.
# NO possession adjustment is applied at inference.

# These are the exact vol_columns from the service (lines 783-793)
VOL_COLS = {
    "passes", "dribbles", "tackles", "possession_won", "possession_lost",
    "fouls_committed", "offsides", "distance_covered", "distance_sprinted",
}
RATE_COLS = {"shot_accuracy", "pass_accuracy", "dribble_success_rate", "tackle_success_rate"}
RARE_COLS = {"goals", "assists", "shots"}  # NOT half-length scaled

time_scalar = svc.H_BASE / half_len
norm        = svc.attr.get("normalized_metrics", {})

print(f"Half-length scalar: H_BASE({svc.H_BASE}) / half_length({half_len}) = {time_scalar:.4f}")
print(f"(Goals, assists, shots: passed through raw — not half-length scaled)")
print(f"(Rate stats: passed through raw — not scaled)")
print(f"(No possession adjustment at inference)")
print()
print(f"{'Stat':<28} {'Raw':>8} {'Type':>12} {'Normalized':>12} {'Svc value':>10}")
print("-" * 75)

all_stats = sorted(VOL_COLS | RARE_COLS | RATE_COLS)
for stat in all_stats:
    raw = PERFORMANCE.get(stat, 0.0)
    if stat in VOL_COLS:
        stat_type = "vol ×time"
        normalized = raw * time_scalar
    elif stat in RARE_COLS:
        stat_type = "rare (raw)"
        normalized = raw
    else:
        stat_type = "rate (raw)"
        normalized = raw
    svc_val = norm.get(stat, float("nan"))
    match_flag = "✓" if abs(normalized - svc_val) < 0.01 else f"≠svc:{svc_val:.3f}"
    print(f"{stat:<28} {raw:>8.3f} {stat_type:>12} {normalized:>12.3f} {match_flag:>10}")


Half-length scalar: H_BASE(10.0) / half_length(10) = 1.0000
(Goals, assists, shots: passed through raw — not half-length scaled)
(Rate stats: passed through raw — not scaled)
(No possession adjustment at inference)

Stat                              Raw         Type   Normalized  Svc value
---------------------------------------------------------------------------
assists                         0.000   rare (raw)        0.000          ✓
distance_covered                6.800    vol ×time        6.800          ✓
distance_sprinted               2.200    vol ×time        2.200          ✓
dribble_success_rate          100.000   rate (raw)      100.000          ✓
dribbles                       12.000    vol ×time       12.000          ✓
fouls_committed                 2.000    vol ×time        2.000          ✓
goals                           0.000   rare (raw)        0.000          ✓
offsides                        0.000    vol ×time        0.000          ✓
pass_accuracy                 100

In [35]:
# ── Step 2: Bayesian smoothing → per-90 rates ─────────────────────────────────

LOG_STATS = {"goals_p90", "assists_p90", "non_goal_shots_p90",
             "offsides_p90", "fouls_committed_p90",
             "possession_won_p90", "possession_lost_p90"}
LOG_PRIOR = {"possession_won", "possession_lost", "fouls_committed", "offsides"}

# Build p90 dict with computed stats that are added after _apply_bayesian_smoothing returns
p90 = dict(svc.attr["p90_metrics"])  # mutable copy
p90["non_goal_shots_p90"] = max(0.0, p90.get("shots_p90", 0.0) - p90.get("goals_p90", 0.0))
# xt_bonus_p90 is computed from multiple inputs; read post-floor capture as best approximation
_post = svc.attr.get("z_scores_at_dot", {})
_ms   = svc.attr.get("pos_means_stds", {})
_xt_ms = _ms.get("xt_bonus_p90", {})
if _xt_ms.get("std", 0) > 0:
    # back-compute xt_bonus_p90 from its z-score
    _xt_z = _post.get("xt_bonus_p90_z", 0.0)
    p90["xt_bonus_p90"] = _xt_z * _xt_ms["std"] + _xt_ms["mean"]
p90.pop("shots_p90", None)  # not a profile stat; remove from display
pos_ms  = svc.attr["pos_means_stds"]

print(f"{'Stat':<28} {'Smoothed p90':>13} {'Log xform?':>11} {'→ log value':>12}")
print("-" * 70)
for stat, val in sorted(p90.items()):
    log_flag = "log(x+1)" if stat in LOG_STATS else ""
    log_val  = f"{math.log1p(val):.4f}" if stat in LOG_STATS else ""
    print(f"{stat:<28} {val:>13.4f} {log_flag:>11} {log_val:>12}")

print()
print("Bayesian smoothing formula (volume stats):")
print("  smoothed_p90 = (raw_count + league_avg × (d/90)) / (minutes + d) × 90")
print()
print("Selected stat detail:")
for stat_name, raw_key in [("passes_p90","passes"),("tackles_p90","tackles"),
                            ("possession_won_p90","possession_won"),("goals_p90","goals")]:
    d     = svc.DUMMY_WEIGHTS.get(raw_key, svc.DEFAULT_DUMMY)
    stored_mean = pos_ms.get(stat_name, {}).get("mean", 0.0)
    import math as _m
    league_avg = _m.expm1(stored_mean) if raw_key in LOG_PRIOR else stored_mean
    raw_count  = svc.attr["normalized_metrics"].get(raw_key, 0.0)
    smoothed   = (raw_count + league_avg * (d / 90.0)) / (minutes + d) * 90.0
    print(f"  {stat_name:<26} raw={raw_count:.3f}  d={d}  "
          f"league_avg={league_avg:.3f}  → {smoothed:.4f} (svc: {p90.get(stat_name, float('nan')):.4f})")

Stat                          Smoothed p90  Log xform?  → log value
----------------------------------------------------------------------
assists_p90                         0.0000    log(x+1)       0.0000
distance_covered_p90               10.4264                         
distance_sprinted_p90               3.3102                         
dribbles_p90                       16.4061                         
fouls_committed_p90                 2.1660    log(x+1)       1.1525
goals_p90                           0.0000    log(x+1)       0.0000
non_goal_shots_p90                  0.0000    log(x+1)       0.0000
offsides_p90                        0.0000    log(x+1)       0.0000
passes_p90                         22.6995                         
possession_lost_p90                 0.1963    log(x+1)       0.1792
possession_won_p90                  7.2978    log(x+1)       2.1160
tackles_p90                         9.0158                         
xt_bonus_p90                        0.6134   

In [36]:
# ── Step 3: Z-scores ──────────────────────────────────────────────────────────

NEG_STATS = {"fouls_committed_p90", "possession_lost_p90", "offsides_p90"}
LOG_STATS = {"goals_p90", "assists_p90", "non_goal_shots_p90",
             "offsides_p90", "fouls_committed_p90", "possession_won_p90", "possession_lost_p90"}

z_scores = svc.attr["z_scores"]   # pre-floor values
pos_ms   = svc.attr["pos_means_stds"]
STAT_COLS = list(svc._PROFILE_COLS)

print(f"{'Stat':<28} {'p90 value':>10} {'→log':>7} {'mean':>8} {'std':>7} "
      f"{'raw_z':>8} {'neg?':>5} {'vol_mask':>9} {'pre_floor_z':>12}")
print("-" * 100)

for col in STAT_COLS:
    z_key = f"{col}_z"
    final_z = z_scores.get(z_key, 0.0)
    p90_val = p90.get(col) if p90.get(col) is not None else svc.attr["normalized_metrics"].get(col, 0.0)
    ms = pos_ms.get(col, {})
    mean = ms.get("mean", 0.0)
    std  = ms.get("std",  1.0)
    log_flag = "✓" if col in LOG_STATS else ""
    log_val  = math.log1p(max(p90_val, 0)) if col in LOG_STATS else p90_val
    raw_z    = ((mean - log_val) / std if col in NEG_STATS else (log_val - mean) / std) if std > 0 else 0.0
    neg_flag = "neg" if col in NEG_STATS else ""
    mask_applied = abs(final_z - raw_z) > 0.001
    mask_str = f"{final_z/raw_z:.3f}×" if (mask_applied and abs(raw_z) > 0.001) else ("0 (low vol)" if mask_applied else "")
    print(f"{col:<28} {p90_val:>10.4f} {log_flag:>7} {mean:>8.4f} {std:>7.4f} "
          f"{raw_z:>8.4f} {neg_flag:>5} {mask_str:>9} {final_z:>12.4f}")

# ── Z-score floors ────────────────────────────────────────────────────────────
pre_floor   = svc.attr["z_scores"]
post_floor  = svc.attr.get("z_scores_at_dot", {})
pos_floors  = dict(svc.Z_SCORE_FLOORS.get(pos, {}))

goals_raw               = PERFORMANCE.get("goals", 0)
non_goal_shots_smoothed = p90.get("non_goal_shots_p90", 0.0)
perf_eff_eligible       = goals_raw >= 1 and non_goal_shots_smoothed == 0.0

has_anything = pos_floors or perf_eff_eligible
print()
print("── Z-score floors " + "─" * 56)

if not has_anything:
    print(f"  No floors defined for position '{pos}'.")
else:
    HDR  = f"  {'Stat':<32} {'Floor':>6}  {'Pre':>8}  {'Post':>8}  Status"
    LINE = f"  {'─'*70}"

    def floor_row(stat_z, floor_val, pre, post):
        hit    = post > pre + 0.0001
        arrow  = "↑ HIT" if hit else "—"
        delta  = f"  (+{post-pre:.4f})" if hit else ""
        return f"  {stat_z:<32} {floor_val:>6.2f}  {pre:>8.4f}  {post:>8.4f}  {arrow}{delta}"

    if perf_eff_eligible:
        print(f"  Perfect efficiency fix (goals≥1 and non_goal_shots==0):")
        print(HDR); print(LINE)
        z_key = "non_goal_shots_p90_z"
        pre   = pre_floor.get(z_key, 0.0)
        post  = post_floor.get(z_key, pre)
        print(floor_row(z_key, 0.0, pre, post))
        print()

    if pos_floors:
        print(f"  Position floors ({pos}):")
        print(HDR); print(LINE)
        for stat_z, floor_val in sorted(pos_floors.items()):
            pre  = pre_floor.get(stat_z, 0.0)
            post = post_floor.get(stat_z, pre)
            print(floor_row(stat_z, floor_val, pre, post))

Stat                          p90 value    →log     mean     std    raw_z  neg?  vol_mask  pre_floor_z
----------------------------------------------------------------------------------------------------
goals_p90                        0.0000       ✓   0.0117  0.0797  -0.1466                      -0.1466
assists_p90                      0.0000       ✓   0.0135  0.0886  -0.1524                      -0.1524
non_goal_shots_p90               0.0000       ✓   0.2429  0.2031  -1.1960                      -1.1960
shot_accuracy                    0.0000          24.9454 15.3565  -1.6244         -0.000×       0.0000
passes_p90                      22.6995          26.4707  9.0700  -0.4158                      -0.4158
pass_accuracy                  100.0000          88.9898 14.1309   0.7792                       0.7791
dribbles_p90                    16.4061          12.1245  5.1010   0.8394                       0.8394
dribble_success_rate           100.0000          87.7367 27.3874   0.4478  

In [37]:
# ── Step 4: Dot product contributions → base_rating ──────────────────────────
# Uses post-floor z-scores (svc.attr["z_scores_at_dot"]) to match what the
# service actually passed into _calculate_dot_product.

pos_weights     = weights.get(pos, {})
dot             = svc.attr["dot_product"]
impact          = svc.attr["pre_modifier"]["impact_scalar"]
z_scores_floored = svc.attr.get("z_scores_at_dot", svc.attr["z_scores"])

print(f"{'Stat':<28} {'Weight':>9} {'Z-score (floored)':>18} {'Contribution':>14}")
print("-" * 73)

contribs = []
for col in STAT_COLS:
    w = pos_weights.get(col, 0.0)
    z = z_scores_floored.get(f"{col}_z", 0.0)
    c = w * z
    contribs.append((col, w, z, c))

contribs.sort(key=lambda x: abs(x[3]), reverse=True)

BAR_HALF = 18
max_abs  = max(abs(c) for _, _, _, c in contribs) or 1.0

for col, w, z, c in contribs:
    sign = "+" if c >= 0 else "-"
    units = int(abs(c) / max_abs * BAR_HALF)
    # Mark stats where floor was applied
    pre_z = svc.attr["z_scores"].get(f"{col}_z", z)
    floored = abs(z - pre_z) > 0.0001
    floor_marker = " F" if floored else "  "
    if c >= 0:
        bar = f"{'':>{BAR_HALF}}|{'█' * units:<{BAR_HALF}}"
    else:
        bar = f"{'█' * units:>{BAR_HALF}}|{'':>{BAR_HALF}}"
    print(f"{col:<28} {w:>9.5f} {z:>18.4f}{floor_marker} {sign}{abs(c):>12.5f}  {bar}")

print("-" * 73)
print(f"{'DOT PRODUCT':<28} {'':>9} {'':>18}   {dot:>14.5f}")
print()
print(f"  F = floor was applied to this z-score")
print()
print(f"Dot product: {dot:.5f}  ({'POSITIVE → impact_scalar NOT applied' if dot >= 0 else 'NEGATIVE → impact_scalar applied'})")
print(f"Impact scalar: √(min({minutes},90)/90) = {impact:.4f}")
adjusted = dot if dot >= 0 else dot * impact
print(f"Adjusted dot:  {adjusted:.5f}")
print()
base_rating = svc.attr["pre_modifier"]["base_rating"]
print(f"BASE RATING = sigmoid({adjusted:.5f}) = {base_rating:.4f}")


Stat                            Weight  Z-score (floored)   Contribution
-------------------------------------------------------------------------
tackles_p90                    0.19926             1.9042   +     0.37943                    |██████████████████
possession_lost_p90            0.15363             1.0689   +     0.16422                    |███████           
fouls_committed_p90            0.04588            -2.6601   -     0.12205               █████|                  
tackle_success_rate            0.12637            -0.9654   -     0.12200               █████|                  
possession_won_p90             0.16407             0.5972   +     0.09798                    |████              
pass_accuracy                  0.04141             0.7791   +     0.03226                    |█                 
dribbles_p90                   0.03464             0.8394   +     0.02907                    |█                 
xt_bonus_p90                   0.02863             1.0037   + 

In [38]:
# ── Step 5: Bonus breakdown ───────────────────────────────────────────────────

pm      = svc.attr["pre_modifier"]
iso     = pm["isolation_multiplier"]
impact  = pm["impact_scalar"]
goals   = pm["goals"]
assists = pm["assists"]
opponent_goals = pm["opponent_goals"]
opp_xg_val     = pm["opponent_xg"]
pos_key = pm["pos"]
mp      = pm["minutes_played"]

bonuses = []

# Goal bonus
alpha = svc.GOAL_ALPHA.get(pos_key, 0.0)
if goals >= 1:
    t_goals = goals * (goals + 1) / 2
    gb = alpha * t_goals * iso
    bonuses.append(
        (
            "Goal bonus",
            f"α={alpha} × T({int(goals)})={int(t_goals)} × iso={iso:.3f}",
            round(gb, 4),
            True,
        )
    )
else:
    bonuses.append(("Goal bonus", "goals=0 — did not fire", 0.0, False))

# Assist bonus
gamma = svc.ASSIST_GAMMA.get(pos_key, 0.0)
if assists >= 1:
    t_assists = assists * (assists + 1) / 2
    ab = gamma * t_assists * iso
    bonuses.append(
        (
            "Assist bonus",
            f"γ={gamma} × T({int(assists)})={int(t_assists)} × iso={iso:.3f}",
            round(ab, 4),
            True,
        )
    )
else:
    bonuses.append(("Assist bonus", "assists=0 — did not fire", 0.0, False))

# Mastery bonuses
print(f"{'Bonus / Condition':<35} {'Detail':<55} {'Fired':>6} {'Amount':>8}")
print("=" * 110)
for name, detail, amount, fired in bonuses:
    flag = "YES" if fired else "no"
    print(f"{name:<35} {detail:<55} {flag:>6} {amount:>+8.4f}")

print()
print("Mastery conditions:")
MASTERY_NAMES = {
    ("tackles_p90_z","possession_won_p90_z"): "Destroyer / Dominant Stopper / Enforcer / Third CB / Two-Way Flank",
    ("passes_p90_z","dribbles_p90_z"):        "Deep-Lying PM / Ball Playing Def / Progression Eng / Wide Playmaker / Complete Fwd",
    ("distance_sprinted_p90_z","xt_bonus_p90_z"): "Express Train / Relentless Engine",
    ("dribbles_p90_z","xt_bonus_p90_z"):      "Direct Threat",
    ("passes_p90_z","xt_bonus_p90_z"):        "Wide Playmaker (Winger/CAM)",
    ("non_goal_shots_p90_z","xt_bonus_p90_z"): "Shadow Striker",
    ("passes_p90_z","tackles_p90_z"):         "Two-Way Engine (WM)",
    ("xt_bonus_p90_z","dribbles_p90_z"):      "Wide Progressor (WM)",
}
for m in svc.attr.get("mastery_log", []):
    key = (m["key_a"], m["key_b"])
    name = MASTERY_NAMES.get(key, f"{m['key_a']} + {m['key_b']}")
    fired_str = "YES" if m["fired"] else "no"
    detail = (f"{m['key_a']}={m['val_a']:.3f}  {m['key_b']}={m['val_b']:.3f}  "
              f"min={m['min_z']:.3f}  thresh={m['threshold']}  excess={m['excess']:.3f}")
    print(f"{name[:35]:<35} {detail:<55} {fired_str:>6} {m['bonus']:>+8.4f}")

# CDM Reliable Pivot
print()
if pos_key == "CDM":
    poss_lost = pm["possession_lost"]
    pass_acc  = pm["pass_accuracy"]
    passes_z  = z_scores.get("passes_p90_z", 0.0)
    gate = mp >= svc.CDM_PIVOT_MIN_MINUTES and pass_acc >= svc.CDM_PIVOT_MIN_PASS_ACC and passes_z > svc.CDM_PIVOT_MIN_PASSES_Z
    print(f"CDM Reliable Pivot gate: mins≥{svc.CDM_PIVOT_MIN_MINUTES}({mp}) AND "
          f"pass_acc≥{svc.CDM_PIVOT_MIN_PASS_ACC}({pass_acc}) AND "
          f"passes_z>{svc.CDM_PIVOT_MIN_PASSES_Z}({passes_z:.3f}) → {'OPEN' if gate else 'CLOSED'}")
    if gate:
        if poss_lost == 0:
            print(f"  Perfect Metronome: poss_lost==0 → +{svc.CDM_PIVOT_PERFECT_METRONOME}")
        elif poss_lost <= 2:
            print(f"  Reliable Shift: poss_lost={poss_lost}≤2 → +{svc.CDM_PIVOT_RELIABLE_SHIFT}")
        else:
            print(f"  poss_lost={poss_lost} > 2 — neither tier fires")

# Clean sheet
print()
cs_ratio = svc.CS_RATIOS.get(pos_key, 0.0)
if opponent_goals == 0 and cs_ratio > 0:
    ramp = min(mp, 60.0) / 60.0
    if opp_xg_val <= 1.0:   tier_val, tier_name = svc.CS_CB_LOW_XG,  f"low xG (≤1.0, xG={opp_xg_val})"
    elif opp_xg_val < 2.0:  tier_val, tier_name = svc.CS_CB_MID_XG,  f"mid xG (1.0-2.0, xG={opp_xg_val})"
    else:                   tier_val, tier_name = svc.CS_CB_HIGH_XG, f"high xG (≥2.0, xG={opp_xg_val})"
    cs_bonus = tier_val * cs_ratio * ramp
    print(f"Clean sheet ({tier_name}): {tier_val} × ratio={cs_ratio} × ramp={ramp:.3f} = +{cs_bonus:.4f}")
elif opponent_goals == 0 and cs_ratio == 0:
    print(f"Clean sheet: opponent scored 0 but {pos_key} has CS ratio=0 — no bonus")
else:
    print(f"Clean sheet: opponent scored {opponent_goals} — no bonus")

print()
total_bonus = svc.attr["total_bonus"]
print(f"TOTAL BONUS: {total_bonus:+.4f}")

Bonus / Condition                   Detail                                                   Fired   Amount
Goal bonus                          goals=0 — did not fire                                      no  +0.0000
Assist bonus                        assists=0 — did not fire                                    no  +0.0000

Mastery conditions:
Destroyer / Dominant Stopper / Enfo tackles_p90_z=1.904  possession_won_p90_z=0.597  min=0.597  thresh=1.2  excess=0.000     no  +0.0000
Deep-Lying PM / Ball Playing Def /  passes_p90_z=-0.416  dribbles_p90_z=0.839  min=-0.416  thresh=1.0  excess=0.000     no  +0.0000


Clean sheet: opponent scored 1.0 — no bonus

TOTAL BONUS: +0.0000


In [39]:
# ── Step 6: Supremacy scalar and final rating ─────────────────────────────────

sup_raw       = svc.attr["supremacy_scalar"]
dot_for_sup   = svc.attr.get("dot_product", 0.0)
quality_factor= max(0.0, 1.0 - dot_for_sup / 1.5)
sup           = sup_raw * quality_factor
base          = svc.attr["pre_modifier"]["base_rating"]
bonus         = svc.attr["total_bonus"]
raw_final     = base + bonus - sup

print("── Supremacy scalar " + "─" * 45)
print(f"  Raw scalar:      {sup_raw:+.4f}  (team_xg={svc.attr['team_xg']}, opp_xg={svc.attr['xg_against']})")
print(f"  Dot product:     {dot_for_sup:+.4f}")
print(f"  Quality factor:  max(0, 1 - {dot_for_sup:.3f}/1.5) = {quality_factor:.4f}")
print(f"  Adjusted:        {sup_raw:.4f} × {quality_factor:.4f} = {sup:+.4f}")
print()
print("── Final calculation " + "─" * 45)
print(f"  base_rating              {base:>+10.4f}")
print(f"  total_bonus              {bonus:>+10.4f}")
print(f"  supremacy (adjusted)     {-sup:>+10.4f}")
print(f"  {'─'*36}")
print(f"  raw final                {raw_final:>+10.4f}")
print(f"  clamped (0–10)           {max(0.0, min(10.0, raw_final)):>10.4f}")
print(f"  ROUNDED                  {round(max(0.0, min(10.0, raw_final)), 1):>10.1f}")
print()
match = "✓" if round(max(0.0, min(10.0, raw_final)), 1) == final_rating else f"≠ service={final_rating}"
print(f"Service output: {final_rating}  {match}")

── Supremacy scalar ─────────────────────────────────────────────
  Raw scalar:      +0.2426  (team_xg=2.7, opp_xg=0.1)
  Dot product:     +0.4863
  Quality factor:  max(0, 1 - 0.486/1.5) = 0.6758
  Adjusted:        0.2426 × 0.6758 = +0.1640

── Final calculation ─────────────────────────────────────────────
  base_rating                 +6.9399
  total_bonus                 +0.0000
  supremacy (adjusted)        -0.1640
  ────────────────────────────────────
  raw final                   +6.7759
  clamped (0–10)               6.7759
  ROUNDED                         6.8

Service output: 6.8  ✓


In [40]:
# ── Report export ─────────────────────────────────────────────────────────────
# Generates a plain-text rating breakdown file you can share without the notebook.

import datetime

report_lines = []
add = report_lines.append

add("=" * 70)
add("RATING ATTRIBUTION REPORT")
add(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
add("=" * 70)
add("")
add("MATCH CONTEXT")
add(f"  Match:       {MATCH_DATA['home_team_name']} vs {MATCH_DATA['away_team_name']}"
    f" ({MATCH_DATA['home_score']}-{MATCH_DATA['away_score']})")
add(f"  Competition: {MATCH_DATA.get('competition', 'N/A')}")
add(f"  Player:      {PERFORMANCE['player_id']}  |  Position: {pos}  |  Minutes: {minutes}")
add(f"  Team xG:     {team_xg}   Opponent xG: {opp_xg}   Opponent goals: {opp_goals}")
add("")

add("NORMALISATION")
add(f"  Half-length scalar: {svc.H_BASE}/{half_len} = {svc.H_BASE/half_len:.3f}")
add(f"  (No possession adjustment applied at inference)")
add("")

add("BAYESIAN SMOOTHING  →  PER-90 RATES")
add(f"  {'Stat':<28} {'Smoothed p90':>13}")
add(f"  {'-'*42}")
for stat, val in sorted(p90.items()):
    add(f"  {stat:<28} {val:>13.4f}")
add("")

add("Z-SCORES")
add(f"  {'Stat':<28} {'p90 value':>10} {'mean':>8} {'std':>7} {'z-score':>9}")
add(f"  {'-'*67}")
for col in STAT_COLS:
    z_key = f"{col}_z"
    final_z = z_scores.get(z_key, 0.0)
    p90_val = p90.get(col) if p90.get(col) is not None else svc.attr["normalized_metrics"].get(col, 0.0)
    ms = pos_ms.get(col, {})
    mean = ms.get("mean", 0.0)
    std  = ms.get("std", 1.0)
    neg  = " (neg)" if col in NEG_STATS else ""
    add(f"  {col:<28} {p90_val:>10.4f} {mean:>8.4f} {std:>7.4f} {final_z:>9.4f}{neg}")
add("")

add("DOT PRODUCT CONTRIBUTIONS  (sorted by absolute contribution)")
add(f"  {'Stat':<28} {'Weight':>9} {'Z-score':>9} {'Contribution':>14}")
add(f"  {'-'*63}")
for col, w, z, c in sorted(contribs, key=lambda x: abs(x[3]), reverse=True):
    sign = "+" if c >= 0 else "-"
    add(f"  {col:<28} {w:>9.5f} {z:>9.4f} {sign}{abs(c):>13.5f}")
add(f"  {'-'*63}")
add(f"  {'DOT PRODUCT':<28} {'':>9} {'':>9} {dot:>+14.5f}")
add("")

_dot_str = "POSITIVE → impact NOT applied" if dot >= 0 else "NEGATIVE → impact applied"
add(f"  Impact scalar: √(min({minutes},90)/90) = {impact:.4f}  [{_dot_str}]")
add(f"  Adjusted dot:  {adjusted:.5f}")
add(f"  BASE RATING:   sigmoid({adjusted:.5f}) = {base_rating:.4f}")
add("")

add("BONUSES")
pm = svc.attr["pre_modifier"]
for name, detail, amount, fired in bonuses:
    status = "FIRED" if fired else "did not fire"
    add(f"  {name:<30} {status:<14} {amount:>+8.4f}  ({detail})")

add("")
add("  Mastery conditions:")
for m in svc.attr.get("mastery_log", []):
    key = (m["key_a"], m["key_b"])
    name = MASTERY_NAMES.get(key, f"{m['key_a']} + {m['key_b']}")
    status = "FIRED" if m["fired"] else "did not fire"
    add(f"  {name[:35]:<35} {status:<14} {m['bonus']:>+8.4f}"
        f"  (min_z={m['min_z']:.3f}, thresh={m['threshold']}, excess={m['excess']:.3f})")

if pos_key == "CDM":
    add("")
    poss_lost = pm["possession_lost"]
    pass_acc  = pm["pass_accuracy"]
    passes_z  = z_scores.get("passes_p90_z", 0.0)
    gate = (minutes >= svc.CDM_PIVOT_MIN_MINUTES and
            pass_acc >= svc.CDM_PIVOT_MIN_PASS_ACC and
            passes_z > svc.CDM_PIVOT_MIN_PASSES_Z)
    add(f"  CDM Reliable Pivot gate: {'OPEN' if gate else 'CLOSED'}")
    add(f"    mins≥{svc.CDM_PIVOT_MIN_MINUTES}: {minutes}  "
        f"pass_acc≥{svc.CDM_PIVOT_MIN_PASS_ACC}: {pass_acc}  "
        f"passes_z>{svc.CDM_PIVOT_MIN_PASSES_Z}: {passes_z:.3f}")

add("")
cs_ratio = svc.CS_RATIOS.get(pos_key, 0.0)
if opp_goals == 0 and cs_ratio > 0:
    ramp = min(minutes, 60.0) / 60.0
    if opp_xg_val <= 1.0:   tv, tn = svc.CS_CB_LOW_XG,  "low xG (≤1.0)"
    elif opp_xg_val < 2.0:  tv, tn = svc.CS_CB_MID_XG,  "mid xG (1.0-2.0)"
    else:                   tv, tn = svc.CS_CB_HIGH_XG, "high xG (≥2.0)"
    cs_bonus = tv * cs_ratio * ramp
    add(f"  Clean sheet ({tn}, xG={opp_xg_val}):  "
        f"{tv} × ratio={cs_ratio} × ramp={ramp:.3f} = {cs_bonus:+.4f}")
elif opp_goals == 0:
    add(f"  Clean sheet: {pos_key} has ratio=0 — no bonus")
else:
    add(f"  Clean sheet: opponent scored {opp_goals} — no bonus")

add("")
add(f"  TOTAL BONUS: {total_bonus:+.4f}")
add("")

add("FINAL CALCULATION")
add(f"  base_rating          {base_rating:>+10.4f}")
add(f"  total_bonus          {total_bonus:>+10.4f}")
add(f"  supremacy_scalar     {-sup:>+10.4f}")
add("  ──────────────────────────────")
add(f"  raw_final            {base_rating + total_bonus - sup:>+10.4f}")
add(f"  FINAL RATING         {final_rating:>10.1f}")
add("")
add("=" * 70)

# Write to file
report_path = project_root / "workshop" / "ratings_creation" / "rating_reports" / \
    f"attribution_{PERFORMANCE['player_id']}_{pos}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_text = "\n".join(report_lines)

with open(report_path, "w") as f:
    f.write(report_text)

print(report_text)
print(f"\nReport saved to: {report_path}")

RATING ATTRIBUTION REPORT
Generated: 2026-08-11 00:04

MATCH CONTEXT
  Match:       Rayo Vallecano vs Valencia CF (1-1)
  Competition: La Liga
  Player:      62  |  Position: CB  |  Minutes: 58
  Team xG:     2.7   Opponent xG: 0.1   Opponent goals: 1

NORMALISATION
  Half-length scalar: 10.0/10 = 1.000
  (No possession adjustment applied at inference)

BAYESIAN SMOOTHING  →  PER-90 RATES
  Stat                          Smoothed p90
  ------------------------------------------
  assists_p90                         0.0000
  distance_covered_p90               10.4264
  distance_sprinted_p90               3.3102
  dribbles_p90                       16.4061
  fouls_committed_p90                 2.1660
  goals_p90                           0.0000
  non_goal_shots_p90                  0.0000
  offsides_p90                        0.0000
  passes_p90                         22.6995
  possession_lost_p90                 0.1963
  possession_won_p90                  7.2978
  tackles_p90          